In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from econml.dml import LinearDML, NonParamDML
from econml.dr import DRLearner, LinearDRLearner
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from xgboost import XGBRegressor, XGBClassifier
from sklearn.linear_model import LassoCV

import matplotlib.pyplot as plt

In [ ]:
def compute_effect(Y, T, X, min_propensity=0.001):
    N = len(Y)
    model_propensity = XGBClassifier()
    model_outcome = XGBRegressor()
    # model = LinearDRLearner(model_propensity=model_propensity, 
    #                         model_regression=model_outcome, 
    #                         min_propensity=min_propensity,
    #                         discrete_outcome=False)
    # model.fit(Y=Y, T=T, X=X); 
    # ate = model.ate(X=X)
    #print(model.ate_interval(X=X))
    model_propensity.fit(X = X,   
                         y = T)
    model_outcome.fit(X = np.concatenate((X, np.expand_dims(T, axis=1)), axis=1),
                      y = Y)
    mu0 = model_outcome.predict(np.concatenate((X, np.zeros((N, 1))), axis=1))
    mu1 = model_outcome.predict(np.concatenate((X, np.ones((N, 1))), axis=1))
    ps = model_propensity.predict_proba(X)[:, 1]
    ps = np.clip(ps, min_propensity, 1-min_propensity)
    norm_1 = np.mean(T/ps)
    norm_0 = np.mean((1-T)/(1-ps))
    ite = mu1-mu0 + T * (Y-mu1) / (ps*norm_1) - (1-T) * (Y-mu0) / ((1-ps)*norm_0)
    ate = np.mean(ite)
    return ate

In [ ]:
p = 0.5
k = 5
N = 10000

results = pd.DataFrame(columns=['N','seed','OS','RCT'])
i = 0
for N in [500, 1000, 2000, 4000, 8000, 16000, 32000]:
    print(f"N={N}")
    for seed in [0,1,2,3,4,5,6,7,8,9]:
        np.random.seed(seed)
        W = np.random.binomial(1, p, N)
        U = np.random.binomial(k, p, N)
        X = np.concatenate((np.expand_dims(W, axis=1), 
                            np.expand_dims(U, axis=1)), 
                            axis=1)

        # RCT
        T = np.random.binomial(1, 0.5, N).astype(int)
        Y = np.round((9*(W/4 + U/(2*k) + T/4) + np.random.binomial(9, 0.5, N))/2).astype(int)
        #Y = (9*(W/4 + U/(2*k) + T/4) + np.random.binomial(9, 0.5, N))/2
        RCT = compute_effect(Y,T,X)

        # OS
        T = np.round((np.random.binomial(3, 0.5, N) + W + U/k)/5).astype(int)
        Y = np.round((9*(W/4 + U/(2*k) + T/4) + np.random.binomial(9, 0.5, N))/2).astype(int)
        #Y = (9*(W/4 + U/(2*k) + T/4) + np.random.binomial(9, 0.5, N))/2
        OS = compute_effect(Y,T,X)
        print(f"Seed={seed}: ATE (RCT)={RCT}, ATE (OS)={OS}")
        results.loc[i] = {'N':N,
                           'seed':seed,
                           'RCT':RCT,
                           'OS':OS
        }
        i += 1

In [ ]:
Ns = results["N"].unique()
OS_mean = results.groupby('N').mean()["OS"]
OS_dev = results.groupby('N').std()["OS"]
RCT_mean = results.groupby('N').mean()["RCT"]
RCT_dev = results.groupby('N').std()["RCT"]
plt.figure(figsize=(10, 5))
plt.xscale("log")
plt.xlim(450,35000)
plt.xticks(Ns,Ns)
plt.errorbar(Ns, RCT_mean, yerr=RCT_dev, label='RCT')
plt.fill_between(Ns, RCT_mean - RCT_dev, RCT_mean + RCT_dev, alpha=0.1)
plt.errorbar(Ns, OS_mean, yerr=OS_dev, label='OS')
plt.fill_between(Ns, OS_mean - OS_dev, OS_mean + OS_dev, alpha=0.1);
plt.xlabel("N")
plt.ylabel("ATE (by AIPW)")
plt.legend()
# ylim
plt.ylim(1, 1.5)

offset = (RCT_mean.iloc[-1]-OS_mean.iloc[-1])/RCT_mean.iloc[-1]*100
print(f'Offset: {offset:.2f}%')


In [ ]:
results

In [ ]:
# train a 2 layer NN from X to Y
import torch
import torch.nn as nn
import torch.nn.functional as F

class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(2, 10)
        self.fc2 = nn.Linear(10, 10)
        self.fc3 = nn.Linear(10, 10)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x
    
# train a 2 layer NN from X to Y
def train_MLP(X, Y, epochs=100):
    model = MLP()
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(epochs):
        optimizer.zero_grad()
        Y_pred = model(torch.tensor(X, dtype=torch.float32))
        loss = criterion(Y_pred, torch.tensor(Y, dtype=torch.float32).long())
        print(f'Epoch {epoch}: loss={loss.item()}')
        loss.backward()
        optimizer.step()
    return model

model = train_MLP(X,Y, epochs=1000)
Y_pred = np.argmax(model(torch.tensor(X, dtype=torch.float32)).detach().numpy(), axis=1)
accuracy = np.mean((Y_pred==Y))
print(f'Accuracy: {accuracy*100:.2f}')

In [ ]:
# train random forest classifier from X to Y
model = RandomForestClassifier()
model.fit(X,Y)
Y_pred = model.predict(X)
accuracy = np.mean((Y_pred==Y))
print(f'Accuracy: {accuracy*100:.2f}')

In [ ]:
def compute_effect(dataset, method="AD", pred=False, total=False, econml=True, train_ratio=1):
    if train_ratio < 1:
        n_tr = int(train_ratio*len(dataset))
        if pred:
            Y = dataset.Y_hat[:n_tr].astype('int')
        else:
            Y = dataset.Y[:n_tr].numpy().astype('int')
        T = dataset.T[:n_tr].numpy().astype('int')
        W = dataset.W[:n_tr].numpy()
        U = dataset.U[:n_tr].numpy()

    else:
        if pred:
            Y = dataset.Y_hat.astype('int')
        else:
            Y = dataset.Y.numpy().astype('int')
        T = dataset.T.numpy().astype('int')
        W = dataset.W.numpy()
        U = dataset.U.numpy()
    N = len(Y)
    if total: 
        if len(W.shape) == 1:
            W = np.expand_dims(W, axis=1)
        if len(U.shape) == 1:
            U = np.expand_dims(U, axis=1)
        X = np.concatenate((W, U), axis=1)
    else:
        if len(W.shape) == 1:
            X = np.expand_dims(W, axis=1)
        else:
            X = W

    if method == "AD":
        return np.mean(Y[T == 1]) - np.mean(Y[T == 0])
    
    # if method == "AF":
    #         return np.mean(Y[(T == 1) & (W == 1)])*np.mean(W==1) + \
    #             np.mean(Y[(T == 1) & (W == 0)])*np.mean(W==0) - \
    #             np.mean(Y[(T == 0) & (W == 1)])*np.mean(W==1) - \
    #             np.mean(Y[(T == 0) & (W == 0)])*np.mean(W==0)
    # if method == "AF_total":
    #     PO_T0 = 0
    #     PO_T1 = 0
    #     for k in range(dataset.k+1):
    #         for w in range(2):
    #             PO_T0 += np.mean(Y[(T == 0) & (U == k) & (W == w)])*np.mean(W == w)*np.mean(U == k)
    #             PO_T1 += np.mean(Y[(T == 1) & (U == k) & (W == w)])*np.mean(W == w)*np.mean(U == k)
    #             # print(f"k={k}, w={w}, PO_T0={PO_T0}, PO_T1={PO_T1}")
    #     return PO_T1 - PO_T0

    if method == "AIPW":
        model_propensity = XGBRegressor()
        model_outcome = XGBRegressor()
        if econml:
            model = LinearDML(model_t=model_propensity, 
                              model_y=model_outcome,
                              discrete_treatment=True,
                              discrete_outcome=False,
                              random_state=1)
            model.fit(Y=Y, T=T, X=X); 
            return model.ate(X=X)
        else:
            model_propensity.fit(X = X,   
                                 y = T)
            model_outcome.fit(X = np.concatenate((X, np.expand_dims(T, axis=1)), axis=1),
                              y = Y)
            mu0 = model_outcome.predict(np.concatenate((X, np.zeros((N, 1))), axis=1))
            mu1 = model_outcome.predict(np.concatenate((X, np.ones((N, 1))), axis=1))
            ps = model_propensity.predict_proba(X)[:, 1]
            norm_1 = np.mean(T/ps)
            norm_0 = np.mean((1-T)/(1-ps))
            ite = mu1-mu0 + T * (Y-mu1) / (ps*norm_1) - (1-T) * (Y-mu0) / ((1-ps)*norm_0)
            return np.mean(ite)